# AI Tutor — **mt100** · Qwen leg · five local arms · representative 100 (`v2`)

Runs the five Jetson-deployable Qwen tutor sizes against the representative-100
multi-turn scenarios (tag `v2`, selected by Task 1, no sampling). This is the
OSS leg of the mt100 board — the cloud leg (Task 5's model list) is scored on
the same 100 scenarios so both legs sit on one combined board.

**Call mode is pinned board-wide.** Judge, student-sim, scenario set, the
`v2` subset AND engine call mode are all held constant across the 19 arms:
Cell 9 below sets `TUTOR_CALL_MODE=two`, and so do the `run_cloud.sh`
invocations for the cloud leg. `_call_mode()` returns a pinned value before it
looks at `family`, so no arm resolves per-family.

`two` matches production's Anthropic configuration and the mt30/mt50 OSS
boards, keeping the 30 `v1` scenarios in this 100 comparable to those runs.
It costs a second LLM call per turn on these five Qwen arms, which `auto`
would have run one-call.

| tag | note |
| --- | --- |
| `qwen3.5-2b-jetson` | hybrid template, think suppressed (`ollama_think=False`) |
| `qwen3-4b-jetson` | instruct checkpoint, nothing to suppress |
| `qwen3-8b-jetson` | hybrid template, think suppressed |
| `qwen3.6-27b-instruct` | hybrid template, think suppressed, `num_ctx=32768` |
| `qwen3-30b-a3b-jetson` | instruct checkpoint, nothing to suppress |

- **Scenarios:** `--multi-turn --subset v2` — the representative 100 multi-turn scenarios
  (tag `v2`), every model sees all 100. No sampling — the subset IS the 100.
- **Tags are Modelfile-pinned, never bare.** run_matrix.sh builds each via
  `ollama create <tag> -f infra/ollama/Modelfile.<tag>` (never `ollama pull`
  a bare tag) — see the module docstring in `_make_colab_nb_mt100.py` for why
  a bare tag silently confounds the run (falls through `get_model_profile`'s
  generic `qwen3` regex to a `num_ctx=24192` CLOUD profile).
- **Tutor** = the local qwen under test; **student-sim + rubric judge** =
  Anthropic (needs `ANTHROPIC_API_KEY`).
- **Branch `offline-harness-copy`** must be pushed and must carry the mt100 profiles/
  Modelfiles — Cell 2 prints HEAD; check it.

**Before you start**
1. Runtime → **Change runtime type** → **T4 GPU** (run_matrix frees weights between models; the 30b-a3b arm is the largest download).
2. Colab Secrets (🔑 sidebar), *Notebook access ON*:
   - `GH_TOKEN` — GitHub classic PAT, `repo` scope (collaborator on `eai6/ai-tutor`).
   - `ANTHROPIC_API_KEY` — **required** (student-sim + rubric judge).
   - `GOOGLE_API_KEY`, `OPENAI_API_KEY` — grader/judge fallback cascade.

## Cell 0 — pick the arm group for THIS tab

Run this notebook in **three tabs**, one per group, to evaluate the five Qwen
arms concurrently. Set the dropdown below before anything else — it decides
which GPU you should pick in Cell 1.

| group | arms | runtime it needs |
| --- | --- | --- |
| `A_small_2b_4b_8b` | `qwen3.5-2b-jetson`, `qwen3-4b-jetson`, `qwen3-8b-jetson` | free T4 is fine |
| `B_27b` | `qwen3.6-27b-instruct` | L4 / A100 — 27B does not fit a T4 |
| `C_30b` | `qwen3-30b-a3b-jetson` | A100 or L4 — ~18 GB at q4, `num_gpu=99` forces full offload |
| `ALL` | all five, sequentially | A100/L4 (because of 30b) |

The groups are disjoint and `run_matrix.sh` takes no lock, so three tabs never
contend — each arm writes its own `<tag>.json` into the same Drive sweep
folder and Cell 10 boards whatever has landed so far. **Do not run the same
group in two tabs**: that races two writers onto one JSON.

Pick `ALL` only if you are running a single tab.

In [ ]:
#@title Arm group for this tab { display-mode: "form" }
GROUP = "A_small_2b_4b_8b"  #@param ["A_small_2b_4b_8b", "B_27b", "C_30b", "ALL"]

ARM_GROUPS = {'A_small_2b_4b_8b': ['qwen3.5-2b-jetson', 'qwen3-4b-jetson', 'qwen3-8b-jetson'], 'B_27b': ['qwen3.6-27b-instruct'], 'C_30b': ['qwen3-30b-a3b-jetson']}
ARMS_THIS_TAB = (
    [t for g in ARM_GROUPS.values() for t in g] if GROUP == "ALL"
    else ARM_GROUPS[GROUP]
)
print(f"this tab runs group {GROUP}: {len(ARMS_THIS_TAB)} arm(s)")
for _t in ARMS_THIS_TAB:
    print("  -", _t)

## Cell 1 — GPU + mount Drive

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — clone the repo (branch `offline-harness-copy` — carries the mt100 profiles/Modelfiles)

In [ ]:
from google.colab import userdata
import subprocess, os
tok = (userdata.get('GH_TOKEN') or '').strip()
assert tok and ' ' not in tok, "GH_TOKEN missing or contains a space — re-save the secret"
url = f"https://{tok}@github.com/eai6/ai-tutor.git"
subprocess.run(['rm', '-rf', '/content/ai-tutor'], check=True)
subprocess.run(['git', 'clone', '--depth', '1', '-b', 'offline-harness-copy', url, '/content/ai-tutor'], check=True)
os.chdir('/content/ai-tutor')
print('cloned at', os.getcwd())
print('HEAD:', subprocess.run(['git','log','-1','--oneline'],capture_output=True,text=True).stdout.strip())

## Cell 3 — fix hardcoded laptop paths

In [ ]:
!sed -i 's#/home/daniel/Documents/work/Nyansapo/web/ai-tutor#/content/ai-tutor#g; s#\$ROOT/venv/bin/python#python#g; s#venv/bin/python#python#g' offline_eval/*.py offline_eval/*.sh

## Cell 4 — install deps + start Ollama
Version-pinned: tool-call parsing is Ollama-version-sensitive (memory: ollama-sweep-gotchas).

In [ ]:
!pip install -q -r requirements.txt
# zstd is REQUIRED: Ollama's Linux artifact is now .tar.zst — install.sh pipes
# through zstd and the legacy .tgz URL 404s for pinned versions (verified
# 2026-08-03: ollama-linux-amd64.tgz?version=0.30.7 -> 404, .tar.zst -> 200).
!apt-get -qq install -y zstd || (apt-get -qq update && apt-get -qq install -y zstd)
import subprocess, time, shutil, os
OLLAMA_VERSION = '0.30.7'
def _has_ollama(): return shutil.which('ollama') is not None
def _install_ollama():
    if _has_ollama(): return True
    for i in range(1, 4):
        print(f'[ollama] pinned install attempt {i}', flush=True)
        subprocess.run(f'curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh', shell=True)
        if _has_ollama(): return True
        time.sleep(5)
    for i in range(1, 4):
        print(f'[ollama] direct-binary attempt {i}', flush=True)
        r = subprocess.run(f'curl -fL --retry 5 --retry-all-errors --connect-timeout 30 '
                           f'-o /tmp/ollama.tar.zst "https://ollama.com/download/ollama-linux-amd64.tar.zst?version={OLLAMA_VERSION}"',
                           shell=True)
        if r.returncode != 0:
            print(f'[ollama] curl exited {r.returncode}', flush=True)
        if os.path.exists('/tmp/ollama.tar.zst') and os.path.getsize('/tmp/ollama.tar.zst') > 1_000_000:
            subprocess.run('tar --zstd -C /usr -xf /tmp/ollama.tar.zst', shell=True)
            if _has_ollama(): return True
        time.sleep(5)
    return False
assert _install_ollama(), "ollama install failed — Runtime -> Disconnect and delete runtime, then retry."
print(subprocess.run(['ollama','--version'],capture_output=True,text=True).stdout.strip())
subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(30):
    if subprocess.run(['bash','-c','ollama list'], capture_output=True).returncode == 0:
        print('ollama ready'); break
    time.sleep(2)
else:
    print('ollama NOT ready — check /content/ollama.log')

## Cell 5 — write `.env` from Colab Secrets

In [ ]:
from google.colab import userdata
open('.env', 'w').write(
    "SECRET_KEY=colab-eval\nDEBUG=True\nEMBEDDING_BACKEND=sqlite\n"
    f"ANTHROPIC_API_KEY={userdata.get('ANTHROPIC_API_KEY')}\n"
    f"GOOGLE_API_KEY={userdata.get('GOOGLE_API_KEY')}\n"
    f"OPENAI_API_KEY={userdata.get('OPENAI_API_KEY')}\n")
print('.env written')

## Cell 6 — fresh DB + eval fixtures

In [ ]:
!python manage.py migrate
!python manage.py loaddata evals/fixtures/institution.json evals/fixtures/lessons.json

## Cell 7 — persist results to Drive (symlink → survives disconnects)
Symlinks `offline_eval/multi_turn_results/mt100/` to `ai-tutor-eval-multiturn/mt100/` on Drive. Resume-safe: run_matrix.sh skips any model that already has a JSON.

In [ ]:
!mkdir -p /content/drive/MyDrive/ai-tutor-eval-multiturn/mt100
!rm -rf offline_eval/multi_turn_results/mt100 && mkdir -p offline_eval/multi_turn_results && ln -s /content/drive/MyDrive/ai-tutor-eval-multiturn/mt100 offline_eval/multi_turn_results/mt100
import os, glob
print('this run writes to:', os.path.realpath('offline_eval/multi_turn_results/mt100'))
done = sorted(os.path.basename(p)[:-5] for p in glob.glob('offline_eval/multi_turn_results/mt100/*.json'))
print('already scored:', done or '(none yet)')

## Cell 8 — write the model list for THIS tab's group
Writes only the arms `GROUP` selected in Cell 0. Column 1 is the BARE tag: run_matrix.sh reads `tag tier`, detects `infra/ollama/Modelfile.<tag>` and builds it via `ollama create` from its registry base — never a bare `ollama pull <tag>` — then prepends `local_ollama/` itself for `TUTOR_MODEL_OVERRIDE`.

In [ ]:
rows = "\n".join(f"{tag:24s} jetson" for tag in ARMS_THIS_TAB)
open('offline_eval/models.txt', 'w').write(
    f"# Qwen mt100 — group {GROUP}, Modelfile-pinned local tags\n{rows}\n")
print(open('offline_eval/models.txt').read())

## Cell 8.5 — sanity: every arm resolves an EXACT profile + has a Modelfile

Guards the single easiest way to silently ruin this run: a misspelled profile
key falls through `get_model_profile`'s generic `qwen3` regex to a CLOUD
profile with `num_ctx=None` (client sizes that to 24192 — the value that
OOMed an 8 GB box in an earlier arm). Each tag below is built with
`ollama create <tag> -f infra/ollama/Modelfile.<tag>` — never a bare
`ollama pull`:

| tag | Modelfile |
| --- | --- |
| `qwen3.5-2b-jetson` | `infra/ollama/Modelfile.qwen3.5-2b-jetson` |
| `qwen3-4b-jetson` | `infra/ollama/Modelfile.qwen3-4b-jetson` |
| `qwen3-8b-jetson` | `infra/ollama/Modelfile.qwen3-8b-jetson` |
| `qwen3.6-27b-instruct` | `infra/ollama/Modelfile.qwen3.6-27b-instruct` |
| `qwen3-30b-a3b-jetson` | `infra/ollama/Modelfile.qwen3-30b-a3b-jetson` |

In [ ]:
import os
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings')
import django; django.setup()
from apps.llm.model_profiles import MODEL_PROFILES

ARMS = [
    ("local_ollama/qwen3.5-2b-jetson", "qwen3.5-2b-jetson"),
    ("local_ollama/qwen3-4b-jetson", "qwen3-4b-jetson"),
    ("local_ollama/qwen3-8b-jetson", "qwen3-8b-jetson"),
    ("local_ollama/qwen3.6-27b-instruct", "qwen3.6-27b-instruct"),
    ("local_ollama/qwen3-30b-a3b-jetson", "qwen3-30b-a3b-jetson"),
]
for spec, tag in ARMS:
    assert spec in MODEL_PROFILES, f"{spec} has NO exact profile entry — would silently fall through to the qwen3 regex fallback (num_ctx=None -> 24192)"
    mf = f"infra/ollama/Modelfile.{tag}"
    assert os.path.exists(mf), f"missing {mf} — this arm would need a bare `ollama pull`, which is forbidden"
    p = MODEL_PROFILES[spec]
    print(f"OK  {spec:<38} num_ctx={p.num_ctx!s:<8} ollama_think={p.ollama_think}   built from {mf}")
print("all five arms: exact profile + Modelfile present")

## Cell 9 — run the eval (100 v2 multi-turn scenarios, all five arms)
Builds each tag from its Modelfile, runs 100 sessions per arm, saves JSON+log to Drive.

`TUTOR_CALL_MODE=two` pins two-call for every arm, matching the pin the cloud leg uses in `MT100_RUNBOOK.md`. Without it these Qwen arms would resolve to ONE-call via `_call_mode`'s per-family `auto` branch while the Anthropic and OpenAI arms ran two-call — different protocols on one board. It also matches the mt30/mt50 OSS boards, so the 30 `v1` scenarios inside this 100 stay comparable to those runs. Cost: two LLM calls per turn instead of one on these five arms.

In [ ]:
!TUTOR_CALL_MODE=two RESULTS_DIR=$PWD/offline_eval/multi_turn_results/mt100 SIMPLE_TUTOR_ENGINE=1 CLEANUP_MODELS=1 \
  MODE="--multi-turn --subset v2" bash offline_eval/run_matrix.sh

## Cell 10 — results: pass rate + session end-reasons + mean rubric

In [ ]:
import json, glob, os
from collections import Counter
def rub(r):
    items = (r.get('rubric_result') or {}).get('items') or []
    app = [i for i in items if i.get('applicable')]
    return sum(i['score'] for i in app)/len(app) if app else None
rows = []
for f in sorted(glob.glob('offline_eval/multi_turn_results/mt100/*.json')):
    d = json.load(open(f)); res = d.get('results') or []
    n = len(res); k = sum(bool(r.get('passed')) for r in res)
    reasons = Counter((r.get('sim_reason') or '?') for r in res)
    scores = [s for s in (rub(r) for r in res) if s is not None]
    mean_rub = sum(scores)/len(scores) if scores else 0.0
    rows.append((os.path.basename(f)[:-5], k, n, mean_rub, dict(reasons)))
print(f"{'MODEL':<22}{'PASS':>8}  {'RUBRIC':>7}   SESSION END-REASONS")
print('-'*76)
for m, k, n, mr, reasons in sorted(rows, key=lambda r: -(r[1]/r[2] if r[2] else 0)):
    print(f"{m:<22}{k:>4}/{n:<3}  {mr:>6.3f}   {reasons}")
if not rows:
    print("(no results yet — run Cell 9)")

## Cell 11 — identity check: did any arm silently think or mis-size its context?
run_matrix.sh's per-model probe writes to `identity.log` in the results dir — grep it here rather than scrolling Cell 9's raw output.

In [ ]:
import os
log = 'offline_eval/multi_turn_results/mt100/identity.log'
if os.path.exists(log):
    print(open(log).read())
else:
    print("(no identity.log yet — run Cell 9)")

## Reading this run

**What "clean" looks like:** all five arms complete 100/100 (or close —
run_matrix.sh logs a skip+reason for anything that fails outright), Cell 8.5
printed `OK` for all five with the expected `num_ctx` per arm (not `None`),
and Cell 11's identity log shows each hybrid arm (`qwen3.5-2b-jetson`,
`qwen3-8b-jetson`, `qwen3.6-27b-instruct`) answering directly rather than
`THINKS` — thinking was suppressed via `ollama_think=False` and should stay
suppressed.

**If an arm shows `num_ctx=None` or a suspiciously fast/garbled run:** stop —
that is the exact confound this board exists to avoid. Check the tag spelling
in `offline_eval/models.txt` against `MODEL_PROFILES` in
`apps/llm/model_profiles.py` character-for-character.

This is the OSS leg only. The cloud leg (14 arms across three vendors, Task 5)
is scored on the same `v2` 100 scenarios via `run_cloud.sh` / a separate
notebook — Task 8 aggregates both legs onto one combined board.

**Call mode is pinned on both legs:** `TUTOR_CALL_MODE=two` in Cell 9 here and
in the `run_cloud.sh` invocations for the cloud leg, so all 19 rows run the
same protocol. Judge, student-sim, scenario set, subset and call mode are all
held constant — the remaining caveats worth carrying into the leaderboard are
the ones in `MT100_RUNBOOK.md`: the gpt-5.6 arms run with reasoning disabled,
the Qwen size ladder mixes generations 3/3.5/3.6, the 30B arm is a capacity
ceiling rather than a deployment candidate, and no OpenAI arm has ever been
through this harness before.